In [1]:
!pip -q install  -U langchain langchain-google-genai requests

In [2]:
import os
os.environ["GOOGLE_API_KEY"]="Google API key"

In [3]:
import json
import requests
from typing import List, Dict

In [4]:
from langchain_core.tools import tool #this is important because we will build or own tools as we go ahead
from langchain_core.messages import HumanMessage, SystemMessage
#SystemMessage will help in defining the behaviour of the LLM and HumanMessage signifies the user query
from langchain_google_genai import ChatGoogleGenerativeAI

In [5]:
#Creating a synthetic datasets

severity_weights = {
    "breathing difficulty": 30,
    "chest pain": 35,
    "high fever": 20,
    "dizziness": 15,
    "mild cough": 5
}

symptom_category_keywords = {
    "respiratory": [
        "cough",
        "breathing difficulty",
        "shortness of breath",
        "chest congestion"
    ],

    "cardiac": [
        "chest pain",
        "palpitations",
        "heart rate",
        "dizziness"
    ],

    "infection": [
        "fever",
        "chills",
        "body ache",
        "fatigue"
    ]
}

### **Healthcare Assessment Tool**

This tool will analyze patient symptoms, age, and other relevant information to provide a healthcare assessment. It calculates a severity score, identifies matched and potentially missing risk indicators, and determines if human intervention is required, all returned in a structured JSON format.

In [6]:
@tool
def healthcare_assessment_tool(patient_symptoms: str, patient_age: int, basic_info: str) -> Dict:
    """
    Analyzes patient symptoms and other basic information to provide a healthcare assessment.

    Args:
        patient_symptoms (str): A comma-separated string of symptoms reported by the patient.
        patient_age (int): The age of the patient.
        basic_info (str): Any other relevant basic information about the patient (e.g., medical history).

    Returns:
        Dict: A dictionary containing the severity score, risk level, matched indicators,
              missing indicators, and whether human intervention is required.
              Example:
              {
                  "severity_score": 75,
                  "risk_level": "HIGH",
                  "matched_indicators": [
                      "chest pain",
                      "breathing difficulty"
                  ],
                  "missing_indicators": [],
                  "human_intervention_required": true
              }
    """
    # Initialize assessment results
    severity_score = 0
    matched_indicators = []
    missing_indicators = []
    human_intervention_required = False

    # Process symptoms
    symptoms_list = [s.strip().lower() for s in patient_symptoms.split(',')]

    # Calculate severity score and identify matched indicators
    for symptom, weight in severity_weights.items():
        if symptom in symptoms_list:
            severity_score += weight
            matched_indicators.append(symptom)

    # Identify potential missing indicators (simplified for this example)
    # This part can be made more sophisticated with NLP techniques
    all_possible_symptoms = set()
    for category, keywords in symptom_category_keywords.items():
        all_possible_symptoms.update([k.lower() for k in keywords])

    for known_symptom in all_possible_symptoms:
        if known_symptom not in symptoms_list and known_symptom in severity_weights.keys():
            # This is a simplification: considering a symptom "missing" if it's in our known list
            # but not in the patient's reported symptoms. In a real system, this would be more nuanced.
            missing_indicators.append(known_symptom)

    # Determine risk level based on severity score
    if severity_score >= 70:
        risk_level = "CRITICAL"
        human_intervention_required = True
    elif severity_score >= 40:
        risk_level = "HIGH"
        human_intervention_required = True
    elif severity_score >= 15:
        risk_level = "MODERATE"
    else:
        risk_level = "LOW"

    # Consider age for human intervention (example logic)
    if patient_age >= 65 and risk_level in ["HIGH", "MODERATE"]:
        human_intervention_required = True

    # Construct the assessment dictionary
    assessment = {
        "severity_score": severity_score,
        "risk_level": risk_level,
        "matched_indicators": matched_indicators,
        "missing_indicators": list(set(missing_indicators) - set(matched_indicators)), # Remove matched from missing
        "human_intervention_required": human_intervention_required
    }

    return  json.dumps(assessment)


In [7]:
tools=[healthcare_assessment_tool]

### **Creating the Gemini Agent**

Now, let's create the Gemini agent. We'll define a system prompt that guides its behavior and ensure it utilizes our `healthcare_assessment_tool` responsibly. The agent will be instructed to provide clear, concise assessments, emphasize that its responses are not medical diagnoses, and flag critical situations.

In [8]:
#using Gemini LLM as the brain for our Agent
llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

In [9]:
!pip install langchain-classic

In [10]:
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Define the system prompt for the agent
system_message_content = (
    "You are a healthcare assessment assistant. Your primary role is to assess patient symptoms "
    "using the provided `healthcare_assessment_tool`. "
    "Always use the tool when structured symptom assessment is required. "
    "Keep your responses short, structured, and easy to understand. "
    "Clearly explain the assessment results without jargon. "
    "NEVER present your response as a definitive medical diagnosis. "
    "Immediately highlight any situations requiring human medical attention or further professional consultation. "
    "Base your response strictly on the information provided by the user. "
    "Do not invent symptoms or medical conditions. "
    "If a user asks for medical advice, gently remind them that you are an AI assistant and cannot provide medical diagnoses, "
    "but you can help assess symptoms based on the information provided."
)

# Create a ChatPromptTemplate
prompt_template = ChatPromptTemplate.from_messages([
    SystemMessage(content=system_message_content),
    MessagesPlaceholder(variable_name="messages"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

# Create the tool-calling agent
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt_template # Pass the ChatPromptTemplate here
)

# Create the AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # Set to True to see the agent's thought process
)


### **Demonstrating the Agent Workflow with a Specific Query**

Let's test the complete agent workflow with the example query provided, where the agent needs to invoke the healthcare assessment tool and then provide a recommendation.

In [11]:
# Patient Query demonstrating the full workflow
print("\n--- Patient Query for Workflow Demonstration ---")
patient_query = (
    "Patient is 62 years old and is experiencing chest pain, "
    "breathing difficulty and dizziness since this morning."
)
print(f"User Input: {patient_query}")

response_workflow = agent_executor.invoke({
    "messages": [
        HumanMessage(content=patient_query)
    ]
})
print("\nAgent's Final Recommendation:")
print(response_workflow["messages"][-1].content)



--- Patient Query for Workflow Demonstration ---
User Input: Patient is 62 years old and is experiencing chest pain, breathing difficulty and dizziness since this morning.


> Entering new AgentExecutor chain...


GooglePermissionDeniedError: Error calling model 'gemini-3.5-flash-lite' (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}

### **Demonstrating the Gemini Agent**

Let's test our newly created agent with a few example patient queries to see how it performs its assessment and adheres to the defined guardrails.

In [ ]:
# Example 1: High severity symptoms
print("\n--- Agent Query 1 ---")
response1 = agent_executor.invoke({
    "messages": [
        HumanMessage(content="I have severe chest pain and difficulty breathing. I'm 50 years old.")
    ]
})
print(response1["messages"][-1].content)

# Example 2: Moderate symptoms with elderly patient
print("\n--- Agent Query 2 ---")
response2 = agent_executor.invoke({
    "messages": [
        HumanMessage(content="I'm feeling dizzy and have a mild cough. I am 72 years old and have a history of high blood pressure.")
    ]
})
print(response2["messages"][-1].content)

# Example 3: Low severity symptoms
print("\n--- Agent Query 3 ---")
response3 = agent_executor.invoke({
    "messages": [
        HumanMessage(content="I have a mild cough and feel a bit tired. I'm 30.")
    ]
})
print(response3["messages"][-1].content)

# Example 4: General query, not directly requiring the tool
print("\n--- Agent Query 4 ---")
response4 = agent_executor.invoke({
    "messages": [
        HumanMessage(content="What are the common symptoms of a cold?")
    ]
})
print(response4["messages"][-1].content)


### Test Cases for Agent Workflow

#### Test Case 1: Low-Risk Scenario

**Example:** Patient is 25 years old and has had a mild cough for two days with no breathing difficulty or chest pain. The agent should identify the situation as relatively low risk and provide an appropriate recommendation.

In [12]:
# Test Case 1: Low-Risk Scenario
print("\n--- Test Case 1: Low-Risk Scenario ---")
patient_query_low_risk = "I am 25 years old and have had a mild cough for two days with no breathing difficulty or chest pain."
print(f"User Input: {patient_query_low_risk}")

response_low_risk = agent_executor.invoke({
    "messages": [
        HumanMessage(content=patient_query_low_risk)
    ]
})
print("\nAgent's Final Recommendation:")
print(response_low_risk["messages"][-1].content)



--- Test Case 1: Low-Risk Scenario ---
User Input: I am 25 years old and have had a mild cough for two days with no breathing difficulty or chest pain.


> Entering new AgentExecutor chain...


GooglePermissionDeniedError: Error calling model 'gemini-3.5-flash-lite' (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}

#### Test Case 2: Moderate-Risk Scenario

**Example:** Patient is 48 years old and has fever, fatigue and persistent cough for four days. The agent should identify the relevant risk indicators and recommend appropriate follow-up.

In [13]:
# Test Case 2: Moderate-Risk Scenario
print("\n--- Test Case 2: Moderate-Risk Scenario ---")
patient_query_moderate_risk = "I am 48 years old and have fever, fatigue and persistent cough for four days."
print(f"User Input: {patient_query_moderate_risk}")

response_moderate_risk = agent_executor.invoke({
    "messages": [
        HumanMessage(content=patient_query_moderate_risk)
    ]
})
print("\nAgent's Final Recommendation:")
print(response_moderate_risk["messages"][-1].content)



--- Test Case 2: Moderate-Risk Scenario ---
User Input: I am 48 years old and have fever, fatigue and persistent cough for four days.


> Entering new AgentExecutor chain...


GooglePermissionDeniedError: Error calling model 'gemini-3.5-flash-lite' (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}

#### Test Case 3: High-Risk Scenario

**Example:** Patient is 67 years old and is experiencing chest pain, breathing difficulty and dizziness. The agent should identify the case as requiring immediate human medical attention.

In [14]:
# Test Case 3: High-Risk Scenario
print("\n--- Test Case 3: High-Risk Scenario ---")
patient_query_high_risk = "I am 67 years old and am experiencing chest pain, breathing difficulty and dizziness."
print(f"User Input: {patient_query_high_risk}")

response_high_risk = agent_executor.invoke({
    "messages": [
        HumanMessage(content=patient_query_high_risk)
    ]
})
print("\nAgent's Final Recommendation:")
print(response_high_risk["messages"][-1].content)



--- Test Case 3: High-Risk Scenario ---
User Input: I am 67 years old and am experiencing chest pain, breathing difficulty and dizziness.


> Entering new AgentExecutor chain...


GooglePermissionDeniedError: Error calling model 'gemini-3.5-flash-lite' (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}

### Extending Use Cases

#### Scenario A: Irrelevant Symptom Category

**Description:** Provide symptoms that do not clearly match any predefined category. The agent should:
* Recognize the limitation.
* Avoid making unsupported conclusions.
* Highlight the need for human assessment where appropriate.

In [ ]:
# Scenario A: Irrelevant Symptom Category
print("\n--- Scenario A: Irrelevant Symptom Category ---")
patient_query_irrelevant = "I am 35 years old and have a strange tingling sensation in my left arm and a feeling of dread."
print(f"User Input: {patient_query_irrelevant}")

response_irrelevant = agent_executor.invoke({
    "messages": [
        HumanMessage(content=patient_query_irrelevant)
    ]
})
print("\nAgent's Final Recommendation:")
print(response_irrelevant["messages"][-1].content)


#### Scenario B: Incomplete Patient Information

**Description:** Provide a query where important information such as age or symptom duration is missing. The agent should identify the missing information instead of assuming values.

In [ ]:
# Scenario B: Incomplete Patient Information
print("\n--- Scenario B: Incomplete Patient Information ---")
patient_query_incomplete = "I have a persistent headache and nausea. I haven't been feeling well for a week. What should I do?"
print(f"User Input: {patient_query_incomplete}")

response_incomplete = agent_executor.invoke({
    "messages": [
        HumanMessage(content=patient_query_incomplete)
    ]
})
print("\nAgent's Final Recommendation:")
print(response_incomplete["messages"][-1].content)


#### Scenario C: Multiple Risk Indicators

**Description:** Provide a patient query containing symptoms from multiple categories. The agent should use the tool appropriately and explain which indicators contributed to the risk assessment.

In [ ]:
# Scenario C: Multiple Risk Indicators
print("\n--- Scenario C: Multiple Risk Indicators ---")
patient_query_multiple = "I am 55 years old and have a high fever, chest pain, and shortness of breath for the past two days."
print(f"User Input: {patient_query_multiple}")

response_multiple = agent_executor.invoke({
    "messages": [
        HumanMessage(content=patient_query_multiple)
    ]
})
print("\nAgent's Final Recommendation:")
print(response_multiple["messages"][-1].content)
